In [0]:
sales_path='development_042_silver_sandbox.demand_forecast.sales_silver'
like_games_path='development_042_silver_sandbox.demand_forecast.clustering_core3_like_games'
time_series_path='development_042_silver_sandbox.demand_forecast.eilers_group_historical_time_series'
run_id='4e21f8afc8db4b179167faebe7fbd0d5'
sales_cl='expr1_sum'
sales_cl_cumsum='cumsum_expr1_sum'
performance=['avg_coin_in_index_vs_house','avg_theo_net_win_index_vs_house']
weight_col=['no_of_slots']
result_dest='development_042_silver_sandbox.demand_forecast.time_series_extrapolated'

In [0]:
sales_df = spark.table(sales_path)
time_series_df = spark.table(time_series_path).filter("own_status = 'owned'")
like_games_df = spark.table(like_games_path)
like_games_df = like_games_df[like_games_df.mlflow_run_id == run_id]
display(like_games_df)
display(sales_df)
display(time_series_df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --- Configuration ---
weight_col_name = weight_col[0]        # 'no_of_slots'
metric_cols = performance + [sales_cl]  # performance indices + sales
curve_length_method = 'min'             # how to determine curve length: 'min', 'max', 'avg'
max_curve_length = 12

# --- Step 1: Build per-like-game monthly time series ---
# Get mapping: test_game_name -> game_name (each test game has ~3 like games)
mapping_df = like_games_df.select("test_game_name", "game_name", "neighbor_rank").dropDuplicates(["test_game_name", "game_name"])

# Get performance time series for like games (game_name, yearmonth, performance, weight)
perf_df = time_series_df.select(
    F.col("game_name"),
    F.col("yearmonth"),
    F.col("own_status"),
    F.col(weight_col_name).cast("double").alias(weight_col_name),
    *[F.col(c).cast("double").alias(c) for c in performance]
)

# Get sales for like games (ep_theme_name, beginning_month_date, expr1_sum)
# Deduplicate sales to one record per (game, month, revenue_type)
unique_sales = sales_df.dropDuplicates(["beginning_month_date", "revenue_type", sales_cl, "ep_theme_name"])
sales_agg = unique_sales.groupBy("ep_theme_name", "beginning_month_date").agg(
    F.sum(F.col(sales_cl).cast("double")).alias(sales_cl)
)

# --- Step 2: Join performance and sales for each like game on date ---
# Outer join so performance months without sales get 0 and we keep full curves
game_ts = perf_df.join(
    sales_agg,
    (perf_df.game_name == sales_agg.ep_theme_name) & (perf_df.yearmonth == sales_agg.beginning_month_date),
    how="left"
).select(
    perf_df.game_name,
    perf_df.yearmonth,
    F.col("own_status"),
    F.col(weight_col_name),
    *[F.col(c) for c in performance],
    F.coalesce(F.col(sales_cl), F.lit(0.0)).alias(sales_cl)
)

# --- Step 3: Map to test games ---
train_data = mapping_df.join(
    game_ts,
    mapping_df.game_name == game_ts.game_name,
    "inner"
).select(
    "test_game_name",
    mapping_df.game_name,
    "neighbor_rank",
    "yearmonth",
    "own_status",
    weight_col_name,
    *[F.col(c) for c in performance],
    F.col(sales_cl)
)

# --- Step 4: Assign ordinal month index (removes calendar alignment) ---
w = Window.partitionBy("test_game_name", "game_name").orderBy("yearmonth")
train_data = train_data.withColumn("month_index", F.row_number().over(w))

# --- Step 5: Exclude like games with zero total sales (no signal) ---
total_sales = train_data.groupBy("test_game_name", "game_name").agg(
    F.sum(sales_cl).alias("total_sales")
)
valid_games = total_sales.filter(F.col("total_sales") > 0).select("test_game_name", "game_name")
train_data = train_data.join(valid_games, ["test_game_name", "game_name"], "inner")

# --- Step 6: Determine curve length per test game ---
game_lengths = train_data.groupBy("test_game_name", "game_name").agg(
    F.max("month_index").alias("game_length")
)
if curve_length_method == 'min':
    curve_lengths = game_lengths.groupBy("test_game_name").agg(F.min("game_length").alias("curve_length"))
elif curve_length_method == 'max':
    curve_lengths = game_lengths.groupBy("test_game_name").agg(F.max("game_length").alias("curve_length"))
else:
    curve_lengths = game_lengths.groupBy("test_game_name").agg(
        F.round(F.avg("game_length")).cast("int").alias("curve_length")
    )
curve_lengths = curve_lengths.withColumn(
    "curve_length",
    F.least(F.col("curve_length"), F.lit(max_curve_length))
)

# Trim to curve length
train_data = train_data.join(curve_lengths, "test_game_name", "inner")
train_data = train_data.filter(F.col("month_index") <= F.col("curve_length"))

# --- Step 7: Weighted aggregation across like games per ordinal month ---
sum_weight = F.sum(weight_col_name)
agg_exprs = [
    F.when(sum_weight > 0, F.sum(F.col(c) * F.col(weight_col_name)) / sum_weight)
     .otherwise(0.0).alias(c)
    for c in metric_cols
]
agg_exprs.append(sum_weight.alias(f"total_{weight_col_name}"))
agg_exprs.append(F.countDistinct("game_name").alias("n_like_games_used"))
agg_exprs.append(F.sort_array(F.collect_set("game_name")).alias("like_games_used"))

predicted_curve = train_data.groupBy("test_game_name", "month_index", "curve_length", "own_status").agg(*agg_exprs)
predicted_curve = predicted_curve.orderBy("test_game_name", "month_index")
predicted_curve = predicted_curve.withColumn("run_id", F.lit(run_id))

# --- Step 8: Cumulative sum of sales over the curve ---
w_cum = Window.partitionBy("test_game_name").orderBy("month_index")
predicted_curve = predicted_curve.withColumn(f"cumsum_{sales_cl}", F.sum(sales_cl).over(w_cum))

print(f"Curve length method: {curve_length_method} | Max curve length: {max_curve_length}")
display(predicted_curve)

In [0]:
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --- Get list of test games and create interactive widget ---
test_games = [row.test_game_name for row in predicted_curve.select("test_game_name").distinct().orderBy("test_game_name").collect()]
dbutils.widgets.dropdown("test_game", test_games[0], test_games, "Select Test Game")
selected_game = dbutils.widgets.get("test_game")

# --- Predicted curve for selected game ---
pred_pd = predicted_curve.filter(F.col("test_game_name") == selected_game).orderBy("month_index").toPandas()
curve_len = int(pred_pd['curve_length'].iloc[0]) if len(pred_pd) > 0 else max_curve_length

# --- Actual data for selected game (if it exists in time series) ---
actual_ts = time_series_df.filter(F.col("game_name") == selected_game)
w = Window.partitionBy("game_name").orderBy("yearmonth")
actual_ts = actual_ts.withColumn("month_index", F.row_number().over(w))

# Join actual performance with actual sales
actual_sales = sales_df.dropDuplicates(["beginning_month_date", "revenue_type", sales_cl, "ep_theme_name"]) \
    .filter(F.col("ep_theme_name") == selected_game) \
    .groupBy("ep_theme_name", "beginning_month_date").agg(F.sum(F.col(sales_cl).cast("double")).alias(sales_cl))

actual_with_sales = actual_ts.join(
    actual_sales,
    (actual_ts.game_name == actual_sales.ep_theme_name) & (actual_ts.yearmonth == actual_sales.beginning_month_date),
    "left"
).select(
    actual_ts.game_name,
    "month_index",
    F.col("no_of_slots").cast("double").alias("no_of_slots"),
    *[F.col(c).cast("double").alias(c) for c in performance],
    F.coalesce(F.col(sales_cl), F.lit(0.0)).alias(sales_cl)
).filter(F.col("month_index") <= curve_len).orderBy("month_index")

actual_pd = actual_with_sales.toPandas()
has_actual = len(actual_pd) > 0

# --- Compute cumulative sum for actual data ---
if has_actual:
    actual_pd[sales_cl_cumsum] = actual_pd[sales_cl].cumsum()

# --- Plot: 4 subplots (coin_in, theo_win, sales, cumulative sales) ---
fig, axes = plt.subplots(4, 1, figsize=(12, 13), sharex=True)
fig.suptitle(f"Actual vs Predicted: {selected_game}", fontsize=14, fontweight='bold')

metrics = [
    ('avg_coin_in_index_vs_house', 'Avg Coin-In Index vs House'),
    ('avg_theo_net_win_index_vs_house', 'Avg Theo Net Win Index vs House'),
    (sales_cl, 'Sales (units)'),
    (sales_cl_cumsum, 'Cumulative Sales (units)')
]

for ax, (col, label) in zip(axes, metrics):
    ax.plot(pred_pd['month_index'], pred_pd[col], 'b-o', markersize=5, label='Predicted (like games)')
    if has_actual and col in actual_pd.columns:
        ax.plot(actual_pd['month_index'], actual_pd[col], 'r--s', markersize=5, label='Actual')
    ax.set_ylabel(label)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

axes[3].set_xlabel('Month Index')

if not has_actual:
    fig.text(0.5, 0.02, '⚠ No actual data found for this test game in the time series', ha='center', fontsize=11, color='orange')

plt.tight_layout()
plt.show()

print(f"Game: {selected_game} | Curve length: {curve_len} | Actual data available: {has_actual}")
if has_actual:
    print(f"Like games used: {pred_pd['like_games_used'].iloc[0]}")